In [ ]:
class Node:

  def __init__(self, name, value=None):
    self.name = name
    self.value = value
    self.domain = []

  def is_assigned(self):
    return self.value != None

  def assign(self, value):
    self.value = value

  def unassign(self):
    self.value = None

  def get_domain(self):
    return self.domain

In [ ]:
class Graph:

  def __init__(self):
    self.adj_list = {}
    self.nodes = {}

  def add_node(self, name, value=None):
    self.nodes[name] = Node(name, value)
    self.adj_list[name] = {}

  def add_edge(self, name_a, name_b, w=1, directed=False):
    if name_a not in self.adj_list.keys(): self.add_node(name_a)
    if name_b not in self.adj_list.keys(): self.add_node(name_b)
    self.adj_list[name_a][name_b] = w
    if not directed: self.adj_list[name_b][name_a] = w

  def get_nodes(self):
    return list(self.nodes.values())

  def get_node_by_name(self, name):
    return self.nodes[name]

  def get_neighbors_of(self, node_name):
    return [self.get_node_by_name(neigh_name) for neigh_name in self.adj_list[node_name].keys()]

In [ ]:
import random
class CSP:

  def __init__(self, X, D, C):
    """
    X: grafo (variables)
    D: lista de valores de posibles que acepta cada nodo
    C: una lista de funciones restrictivas
    """
    self.X = X
    self.D = D
    self.C = C
    for node in self.X.get_nodes():
      node.domain = D.copy()

  def is_complete(self):
    return all([node.is_assigned() for node in self.X.get_nodes()])

  def get_unassigned_variables(self):
    return [node for node in self.X.get_nodes() if not node.is_assigned()]

  def get_assigned_variables(self):
    return [node for node in self.X.get_nodes() if node.is_assigned()]

  def select_unassigned_variable(self):
    return random.choice(self.get_unassigned_variables())

  def is_arc_consistent(self, var_i, var_j):
    return all([constraint(var_i, var_j) for constraint in self.C])

  def is_var_consistent(self, var):
    return all([self.is_arc_consistent(var, neigh) for neigh in self.X.get_neighbors_of(var.name) if neigh.is_assigned()])

  def get_variables_values(self):
    return {node.name: node.value for node in self.X.get_nodes()}

In [ ]:
class BinairoCSP(CSP):

  def __init__(self, size=(6,6)):
    self.n_rows, self.n_columns = size
    assert self.n_rows == self.n_columns, "Rows and columns must be equal"
    assert self.n_rows % 2 == 0, "Rows must be divisible by 2"
    assert self.n_columns % 2 == 0, "Columns must be divisible by 2"
    self.columns = [str(i) for i in range(self.n_columns)]
    self.rows = [str(i) for i in range(self.n_rows)]
    self.beautiful_mapping = {"0": "⬜", "1": "⬛", None: "🔹"}
    super().__init__(self.create_board(), ["0", "1"], [self.no_triplets])

  def create_board(self):
    g = Graph()
    for cell_name in self.get_all_possible_cell_names():
      g.add_node(cell_name)

    for i in range(self.n_rows):
      for j in range(self.n_columns):
        node = f"{i}_{j}"
        # Column neighbors (distance 1 and 2)
        for d in [1, 2]:
          if j + d < self.n_columns:
            neigh = f"{i}_{j+d}"
            g.add_edge(node, neigh)
            g.add_edge(neigh, node)
        # Row neighbors (distance 1 and 2)
        for d in [1, 2]:
          if i + d < self.n_rows:
            neigh = f"{i+d}_{j}"
            g.add_edge(node, neigh)
            g.add_edge(neigh, node)

    return g

  def get_all_possible_cell_names(self):
    cells = []
    for column in self.columns:
      for row in self.rows:
        cells.append(column + "_" + row)
    return cells

  def assign_value_to_cell(self, cell_name, value):
    node = self.X.get_node_by_name(cell_name)
    node.assign(value)

  def show_board(self):
    for idx, node in enumerate(self.X.get_nodes(), 1):
      val = self.beautiful_mapping[node.value]
      print(val, "", end="")
      if(idx % self.n_columns == 0): print()

  def read_txt(self, filename):
    with open(filename) as file:
      for row, line in enumerate(file.readlines()):
        for col, val in enumerate(line.replace("\n", "")):
          cell_name = self.columns[col] + "_" + self.rows[row]
          val = val if val != "-" else None
          self.assign_value_to_cell(cell_name, val)

  def no_triplets(self, var_i, var_j):

    i, j = map(int, var_i.name.split("_"))
    ni, nj = map(int, var_j.name.split("_"))

    # Different rows or columns: irrelevant for the "no three" rule
    if i != ni and j != nj:
      return True

    # If either unassigned, no violation yet
    if var_i.value is None or var_j.value is None:
      return True

    # Check same row
    if i == ni:
      # Horizontal case
      if var_i.value == var_j.value:
        # Check if there is a same-value neighbor before or after
        left = self.X.get_node_by_name(f"{i}_{j-1}") if j-1 >= 0 else None
        right = self.X.get_node_by_name(f"{i}_{nj+1}") if nj+1 < self.n_columns else None
        if (left and left.value == var_i.value) or (right and right.value == var_i.value):
          return False
    else:
      # Vertical case
      if var_i.value == var_j.value:
        up = self.X.get_node_by_name(f"{i-1}_{j}") if i-1 >= 0 else None
        down = self.X.get_node_by_name(f"{ni+1}_{j}") if ni+1 < self.n_rows else None
        if (up and up.value == var_i.value) or (down and down.value == var_i.value):
          return False

    return True

  def is_constraint_satisfied(self, var_i, val_i, var_j, val_j):
    # Temporarily assign values to test constraint
    old_i, old_j = var_i.value, var_j.value
    var_i.value, var_j.value = val_i, val_j
    ok = self.is_arc_consistent(var_i, var_j)
    var_i.value, var_j.value = old_i, old_j
    return ok

  def save_domains(self):
    return {node.name: node.get_domain()[:] for node in self.X.get_nodes()}

  def restore_domains(self, backup):
    for node in self.X.get_nodes():
      node.domain = backup[node.name]


In [ ]:
from collections import deque

class AC3Solver:
  def __init__(self, csp):
    self.csp = csp

  def ac3(self):
    # Initialize queue with all arcs (pairs of variables with constraints)
    queue = deque()
    for xi in self.csp.X.get_nodes():
      for xj in self.csp.X.get_neighbors_of(xi.name):
        queue.append((xi, xj))

    while queue:
      xi, xj = queue.popleft()
      if self.revise(xi, xj):
        # If domain wiped out, inconsistency
        if not xi.get_domain():
          return False
        # Add back all neighbors except xj
        for xk in self.csp.X.get_neighbors_of(xi.name):
          if xk != xj:
            queue.append((xk, xi))
    return True

  def revise(self, xi, xj):
    revised = False
    for x in xi.get_domain()[:]:  # copy since we may modify
      # Keep x only if ∃y ∈ Dj that satisfies constraint
      if not any(
        self.csp.is_constraint_satisfied(xi, x, xj, y)
        for y in xj.get_domain()
      ):
        xi.remove_from_domain(x)
        revised = True
    return revised


In [ ]:
class BacktrackSolver:
  def __init__(self, csp, with_ac3=True):
    self.csp = csp
    self.ac3_solver = AC3Solver(csp)
    self.with_ac3 = with_ac3

  def solve(self):
    if self.with_ac3 and not self.ac3_solver.ac3():
      return False
    return self.backtrack()

  def backtrack(self):

    if self.csp.is_complete():
      return True
    var = self.csp.select_unassigned_variable()

    for value in var.get_domain():
      var.assign(value)
      if self.csp.is_var_consistent(var):
        backup = self.csp.save_domains()
        if not self.with_ac3 or self.ac3_solver.ac3():
          result = self.backtrack()
          if result:
            return result
        self.csp.restore_domains(backup)
      var.unassign()
    return False


In [ ]:
binairo = BinairoCSP()
binairo.read_txt("binairo.txt")

In [ ]:
binairo.show_board()

🔹 🔹 🔹 🔹 🔹 🔹 
🔹 🔹 ⬛ 🔹 🔹 🔹 
🔹 🔹 ⬛ 🔹 🔹 🔹 
🔹 🔹 🔹 🔹 🔹 🔹 
🔹 🔹 🔹 🔹 🔹 🔹 
🔹 🔹 🔹 🔹 🔹 🔹 


In [ ]:
solver = BacktrackSolver(binairo, with_ac3=True)
solver.solve()

True

In [ ]:
binairo.show_board()

⬛ ⬜ ⬜ ⬛ ⬜ ⬜ 
⬜ ⬛ ⬛ ⬜ ⬜ ⬛ 
⬜ ⬛ ⬛ ⬜ ⬛ ⬜ 
⬛ ⬜ ⬜ ⬛ ⬜ ⬛ 
⬜ ⬛ ⬛ ⬜ ⬛ ⬜ 
⬛ ⬜ ⬜ ⬛ ⬜ ⬛ 
